# Minilink showcase

Minilink is a Python-native block-diagram framework for modeling, simulating,
optimizing, and visualizing dynamical systems. The promise: **the code reads
like the textbook math**, and **everything is a `System`** — plants,
controllers, sources, and full diagrams alike.

This notebook is a compact guided tour:

1. [Quick start](#1.-Quick-start) — phase plane of **f**, simulate, open/closed loop
2. [Everything is a System](#2.-Everything-is-a-System)
3. [Write your own model](#3.-Write-your-own-model)
4. [Compose diagrams](#4.-Compose-diagrams) — operators and autowire
5. [Plant and control catalogs](#5.-Plant-and-control-catalogs) — forced response and LQR
6. [Animation renderers](#6.-Animation-renderers) — animate the LQR cart-pole
7. [Optimization](#7.-Optimization) — pure NLP demo (`MathematicalProgram`)
8. [Compiled execution and JAX](#8.-Compiled-execution-and-JAX) — `JaxCartPole` compile + autodiff
9. [Trajectory optimization](#9.-Trajectory-optimization) — swing-up with JAX gradients

**Run it locally** with minilink installed (`pip install -e .` from the repo
root), or uncomment the Colab setup cell below.


In [ ]:
# Setup for Google Colab (uncomment to run there):
# !git clone https://github.com/alx87grd/minilink
# import sys
# sys.path.append('/content/minilink')

## 1. Quick start

One plant, three levels: visualize **f**, simulate, drive it open-loop, then
close the loop with a controller.

### A plant by itself

Every minilink model is a **`System`**: named input/output **ports**, a state
vector `x`, and parameters `params`. The contract is one **dynamics** equation
plus one map **per output port**:

    dx_dt = f(x, u, t, params)          # how the state evolves 
    y     = h(x, u, t, params)          # one equation per output port

Catalog plants and custom subclasses implement `f` and the port output maps in
textbook NumPy style. Pick a plant, set `params` and `x0`, and display the
object — its port diagram shows the boundary signals.

In [ ]:
import numpy as np

from minilink.dynamics.catalog.pendulum.pendulum import Pendulum

sys = Pendulum()
sys.params["l"] = 1.2  # arm length [m]
sys.params["m"] = 1.0  # mass [kg]
sys.params["d"] = 0.3  # damping [N.s/m]

sys

Visualize **f**: `plot_phase_plane()` draws the
vector field of the dynamics — how the state would move at each point.

In [ ]:
sys.plot_phase_plane()

`compute_trajectory()` integrates **f** and caches the result;
`plot_trajectory()` stacks labeled, unit-aware signals. Re-run the phase plane
to overlay the simulated path.

In [ ]:
sys.x0[0] = 2.0  # initial angle [rad]
sys.compute_trajectory(tf=10.0)
sys.plot_trajectory()
sys.plot_phase_plane()

### Open loop: `source >> plant`

The `>>` operator chains a step source into the pendulum input. The diagram is
itself a `System`.

In [ ]:
from minilink.blocks.sources import Step

step = Step()
step.params["initial_value"] = np.array([0.0])
step.params["final_value"] = np.array([2.0])
step.params["step_time"] = 5.0

diagram = step >> sys
diagram 

In [ ]:
diagram.compute_trajectory()
diagram.plot_trajectory()

### Closed loop: `controller @ plant`

The `@` operator builds the standard feedback loop. The reference port `r` is
left at its nominal zero here.

In [ ]:
from minilink.control.linear import PDController

ctl = PDController()
ctl.params["Kp"] = 100.0
ctl.params["Kd"] = 10.0

diagram2 = step >> ctl @ sys

diagram2

In [ ]:
diagram2.compute_trajectory(tf=10.0)
diagram2.plot_trajectory()

## 2. Everything is a System

A `System` owns its equations (`f`, `h`), named **ports**, default
**parameters**, and initial condition. Trajectories live in the returned
`Trajectory` — not hidden inside the object.

In [ ]:
print("Plant:   ", sys.name)
print("states:  ", sys.n, sys.state.labels, sys.state.units)
print("inputs:  ", sys.m, list(sys.inputs))
print("outputs: ", list(sys.outputs))
print("params:  ", sys.params)
print("Diagram: ", diagram2.n, sys.state.labels, sys.state.units)

## 3. Write your own model

Subclass `DynamicSystem` and implement `f(x, u, t, params)` in textbook style.

In [ ]:
from minilink.core.system import DynamicSystem


class MassSpringDamper(DynamicSystem):
    # m x'' + c x' + k x = u

    def __init__(self):
        super().__init__(n=2, input_dim=1, expose_state=True)
        self.params = {"m": 1.0, "k": 4.0, "c": 0.3}
        self.state.labels = ["x", "dx"]

    def f(self, x, u, t=0, params=None):
        params = self.params if params is None else params
        m, k, c = params["m"], params["k"], params["c"]
        x, dx = x
        f = u[0]
        ddx = (f - c * dx - k * x) / m
        return np.array([dx, ddx])


msd = MassSpringDamper()
msd.x0[0] = 1.0
msd_chain = Step(final_value=np.array([10.0]), step_time=2.0) >> msd
msd_chain.compute_trajectory(tf=20.0, n_steps=1001)
msd_chain.plot_trajectory()


## 4. Compose diagrams

| Shortcut | Meaning |
| --- | --- |
| `a + b + c` | add subsystems without wiring |
| `source >> plant` | chain output to input |
| `controller @ plant` | simple feedback diagram |
| `.autowire(strict=True)` | connect matching named ports |

When port names are unambiguous, autowire builds the whole loop in one line:

In [ ]:
auto = Step() + PDController() + Pendulum()
auto

In [ ]:
auto.autowire(strict=True)
auto

## 5. Plant and control catalogs

`minilink.dynamics.catalog` ships ready-to-use plants; `minilink.control` ships ready-to-wire controllers and design
factories. Below: forced cart-pole response, then LQR stabilization.


In [ ]:
from minilink.dynamics.catalog.pendulum.cartpole import CartPole

cartpole = CartPole()
cartpole.compute_forced(lambda t: np.array([3.0 * np.sin(2.0 * t)]))
cartpole.plot_trajectory()

### Control catalog

Controllers are `System` blocks with standard ports (`r`, `y`, `u`).
The quick start used `PDController`; the catalog also includes
`PIDController`, `FilteredPIDController`, and LQR design helpers such as
`lqr_at_operating_point` — linearize at an equilibrium and return a
`LinearStateFeedbackController` ready to wire with `@`.


In [ ]:
from minilink.control.lqr import lqr_at_operating_point

plant = CartPole()
x_bar = np.array([0.0, np.pi, 0.0, 0.0])
Q = np.diag([1.0, 1.0, 1.0, 1.0])
R = np.array([[1.0]])

lqr_ctl = lqr_at_operating_point( plant, x_bar, Q, R)

lqr_loop = lqr_ctl @ plant

plant.x0 = np.array([-3.0, np.pi - 0.3, 0.0, 0.0])
lqr_loop.compute_trajectory(tf=8.0)
lqr_loop.plot_trajectory()

In [ ]:
lqr_loop

## 6. Animation renderers

`animate()` replays a cached trajectory as geometry attached to the
system. The default renderer draws inline in notebooks; other backends
include `plotly`, `meshcat`, and `pygame`. Uncomment one line at a time
(behavior differs in Colab vs local).


In [ ]:
lqr_loop.animate()
# lqr_loop.animate(renderer="plotly")    # interactive plots in the notebook
# lqr_loop.animate(renderer="meshcat")   # 3D playback in the browser
# lqr_loop.animate(renderer="pygame")    # native window (local scripts)


## 7. Optimization

`minilink.optimization` solves finite-dimensional NLPs: define a
`MathematicalProgram` (`J`, optional constraints), then `Optimizer`.
Trajectory planning (§9) transcribes a `PlanningProblem` into the same layer.

Minimal unconstrained quadratic — min ½‖z − z̄‖², solution z* = z̄:


In [ ]:
import numpy as np

from minilink.optimization.mathematical_program import MathematicalProgram
from minilink.optimization.optimizer import Optimizer

z_bar = np.array([1.0, -0.5, 2.0])

prog = MathematicalProgram(
    n_z=3,
    J=lambda z: 0.5 * np.dot(z - z_bar, z - z_bar),
    grad_J=lambda z: z - z_bar,
)

res = Optimizer(prog, z0=np.zeros(3), method="ipopt"
).solve(disp=True)


## 8. Compiled execution and JAX

Catalog JAX twins such as `JaxCartPole` compile to a flat execution plan.
`compile(backend="jax")` JIT-compiles **f** and keeps it traceable for
autodiff.


In [ ]:
import time

import jax
from minilink.core.backends import configure_jax
from minilink.dynamics.catalog.pendulum.cartpole import JaxCartPole
# from minilink.dynamics.catalog.pendulum.cartpole import CartPole

configure_jax(enable_x64=True)

jax_cartpole = JaxCartPole()
# cartpole = CartPole()

jax_eval = jax_cartpole.compile(backend="jax")


n = 1000
xc = np.zeros(jax_cartpole.n)
uc = np.zeros(jax_cartpole.m)

t0 = time.perf_counter()
for _ in range(n):
    # cartpole.f(xc, uc, 0.0)
    jax_cartpole.f(xc, uc, 0.0)
t_interp = time.perf_counter() - t0

t0 = time.perf_counter()
for _ in range(n):
    jax_eval.f(xc, uc, 0.0)
t_jit = time.perf_counter() - t0

print(f"JaxCartPole.f (interpreted) : {1e6 * t_interp / n:6.1f} µs/call")
print(f"compiled (jax jit)          : {1e6 * t_jit / n:6.1f} µs/call  ({t_interp / t_jit:.0f}× faster)")

A = jax.jacfwd(lambda x: jax_eval.f(x, uc, 0.0))(xc)
print("Linearization A = df/dx:\n", np.asarray(A).round(2))


## 9. Trajectory optimization

Cart-pole swing-up on `jax_cartpole` — `compile_backend="jax"` reuses the same
JIT-compiled **f** for dynamics defects and NLP gradients.


In [ ]:
from minilink.core.costs import QuadraticCost
from minilink.planning.problems import PlanningProblem
from minilink.planning.trajectory_optimization.direct_collocation import (
    DirectCollocationOptions,
    DirectCollocationTranscription,
)
from minilink.planning.trajectory_optimization.planner import (
    TrajectoryOptimizationOptions,
    TrajectoryOptimizationPlanner,
)

jax_cartpole.inputs["u"].lower_bound[0] = -10.0
jax_cartpole.inputs["u"].upper_bound[0] = 10.0

x_start = np.array([-2.0, 0.0, 0.0, 0.0])
x_goal = np.array([0.0, np.pi, 0.0, 0.0])

problem = PlanningProblem(
    sys=jax_cartpole,
    x_start=x_start,
    x_goal=x_goal,
    cost=QuadraticCost.from_system(
        jax_cartpole, Q=np.diag([1.0, 1.0, 0.0, 0.0]), xbar=x_goal
    ),
)

planner = TrajectoryOptimizationPlanner(
    problem,
    transcription=DirectCollocationTranscription(
        DirectCollocationOptions(tf=4.0, n_steps=20)
    ),
    options=TrajectoryOptimizationOptions(
        compile_backend="jax",
        optimizer_method="ipopt",  # fallback: "scipy_slsqp"
        solve_disp=True,
    ),
)

traj = planner.compute_solution()
planner.plot_solution(signals=("x", "u"))


In [ ]:
jax_cartpole.animate( traj )